In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.style.use('default')
mpl.rcParams['axes.linewidth'] = 7  # set the value globally
mpl.rcParams['xtick.major.size'] = 24
mpl.rcParams['xtick.major.width'] = 7
mpl.rcParams['xtick.minor.size'] = 16
mpl.rcParams['xtick.minor.width'] = 7
mpl.rcParams['ytick.major.size'] = 24
mpl.rcParams['ytick.major.width'] = 7
mpl.rcParams['ytick.labelsize'] = 75
mpl.rcParams['xtick.labelsize'] = 75
mpl.rcParams['ytick.minor.size'] = 16
mpl.rcParams['ytick.minor.width'] = 7
mpl.rcParams['font.size'] = 55
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['svg.fonttype'] = 'none'

In [ ]:
file ='Tile_Ef_violinplots.csv'

df = pd.read_csv(file)

In [ ]:

N=12

fig, ax = plt.subplots(figsize=(30, 15))
colors = iter(plt.cycler("color", plt.cm.YlOrRd(np.linspace(0.8, 0.25, N))))
basal_cell_df = df
basal_cell_df = basal_cell_df.loc[(basal_cell_df['timepoint'] == 4) & (basal_cell_df['experimentparameter'] ==  'Growth')]

uni_construct = basal_cell_df['construct'].unique()
construct_array = sorted(np.array(uni_construct).astype(int))


func_1 = np.vectorize(lambda x : ((x-1) *20 + 1))
func_2 = np.vectorize(lambda x : ((x) * 20 + 44))

pos_1 = func_1(construct_array)
pos_2 = func_2(construct_array)

xlabel_list = []
for i, pos in enumerate(pos_1):
    '''
    start = pos_1[i]
    end = pos_2[i]
    sequence = f'{start}-{end}'
    '''
    sequence = f'T{i+1}'
    xlabel_list.append(sequence)


basal_generator = basal_cell_df.groupby(['construct', 'date', 'well'], observed=True)
medians = basal_generator['Efret'].median()
for index, df_container in enumerate(basal_generator):
    if index == 0:
        #select first color
        color = next(colors)['color']
        #initiate positions for plots
        position = 1
        #set current_construct to change positions
        current_construct = df_container[0][0]
        #initiate list for x tick label axis
        x_tick_list = [current_construct]
        
        # set up statistics to plot
        current_medians = medians[current_construct]
        experiment_median = current_medians.median()
        experiment_std = current_medians.std()
        plot_stats = False
        
    if current_construct != df_container[0][0]:
        color = next(colors)['color']
        position += 1
        current_construct = df_container[0][0]
        x_tick_list.append(current_construct)
        current_medians = medians[current_construct]
        experiment_median = current_medians.median()
        experiment_std = current_medians.std()
        plot_stats = False
    if not plot_stats:
        ax.scatter(y=experiment_median, x=[position], zorder=3, color='gray', s=1000, edgecolor='k' )
        
        x_positions = position + np.random.uniform(-0.1, 0.1, size=len(current_medians))
        ax.scatter(y = current_medians, x=  x_positions, s=450, edgecolor='Navy', linewidth=6, c='white', alpha=0.3)
        #ax.text(0.68, position, s='n='+str(len(current_medians)), ha ='center')
        ax.errorbar(x=[position], y=experiment_median, yerr=experiment_std, color='k', elinewidth=8, zorder=3)
        #ax.scatter(x=[position], y=experiment_median, zorder=3, color='gray', s=350 )
        #ax.scatter(x = [position] * len(current_medians), y= current_medians, s=300, edgecolor='navy', linewidth=6, c='white')
        #ax.text(position, 0.28, s='n='+str(len(current_medians)), ha ='center')
        
        plot_stats = True
    basal_df = df_container[1]
    data = basal_df['Efret']
    violins = ax.violinplot(data, [position], widths=0.98, showextrema=False, showmedians=False, showmeans=False)

    for pc in violins['bodies']:
        pc.set_facecolor('None')
        pc.set_edgecolor(color)
        pc.set_linewidth(8)
        pc.set_alpha(0.3)


ax.set_xticks(np.arange(1, len(x_tick_list)+1, 1))

ax.set_xticklabels(xlabel_list, fontsize=65)    
#ax.set_ylabel('Efret', fontsize='xx-large')
#ax.set_xlabel('Construct', fontsize='xx-large')
ax.set_ylim([0.25, 0.65])
ax.set_yticks([0.3, 0.4, 0.5, 0.6])
plt.savefig(r'basal_dim_violins_wNewData.svg', format='svg', bbox_inches='tight')
plt.show()